# Going further: the Shockley–Queisser limit {#sec-sq}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/sq-limit.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter5/notebooks/sq-limit.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter5/detlib.py', 'detlib.py')
else:
    sys.path.insert(0, '..')                 # detlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import detlib as dl
dl.use_style()

*This chapter is not in the slides. Slide 130 lists record efficiencies of 25–32 %; here we ask why no single-junction cell can do
much better, using the black-body radiation and detailed balance of Ch. 4.*

## Key relations

Treat the Sun as a black body at $T_s\approx5800$ K seen under a small solid angle, and the cell as a semiconductor of band gap $E_g$ at
$T_c=300$ K. Shockley and Queisser (1961) made four assumptions: every photon with $h\nu\ge E_g$ is absorbed and gives exactly one
electron–hole pair; photons below $E_g$ are lost; the carriers relax to the band edges (the excess energy is lost as heat); and the only
recombination is **radiative**, as required by detailed balance. The photon flux of a black body per unit photon energy, emitted into a
hemisphere, is

$$
\phi_{bb}(E,T)=\frac{2\pi}{h^3c^2}\frac{E^2}{e^{E/k_BT}-1}.
$$

The Sun is diluted by the factor $f=(R_\odot/d_\text{Earth})^2\approx2.16\times10^{-5}$ (the solid angle of the solar disc over π), so

$$
J_{sc}=ef\int_{E_g}^\infty\phi_{bb}(E,T_s)\,dE,\qquad J_0=e\int_{E_g}^\infty\phi_{bb}(E,T_c)\,dE .
$$

In the dark the cell must emit as many photons as it absorbs from its 300 K surroundings: that radiative emission is the dark current
$J_0$, and under bias it grows as $e^{eV/k_BT_c}$. The I–V curve is that of an ideal diode, $J=J_{sc}-J_0(e^{eV/k_BT_c}-1)$, and the
efficiency is the maximum of $JV$ divided by the incident power $f\sigma T_s^4\approx1360$ W/m². A small gap absorbs many photons but
gives a low voltage; a large gap gives a high voltage but wastes most photons. The optimum, about **30.5 % at $E_g\approx1.3$ eV** for
black-body sunlight (33.7 % for the real AM1.5 spectrum at 1.34 eV), is the Shockley–Queisser limit.

## Live panel: efficiency versus band gap and where the energy goes

```{ojs}
//| echo: false
import {shockleyQueisser, kT, qe, h, c0} from "./det_sim.js"
viewof sq = Inputs.form({
  Eg: Inputs.range([0.5, 3], {value: 1.12, step: 0.01, label: "band gap E_g (eV)"}),
  Ts: Inputs.range([3000, 7000], {value: 5778, step: 10, label: "Sun temperature T_s (K)"}),
  Tc: Inputs.range([200, 400], {value: 300, step: 1, label: "cell temperature T_c (K)"}),
  lX: Inputs.range([0, 4.67], {value: 0, step: 0.01, label: "log₁₀ concentration X (max 46 200)"})
})
```

```{ojs}
//| echo: false
sqOut = {
  const X = 10 ** sq.lX, curve = d3.range(0, 126).map(i => { const Eg = 0.5 + 2.5 * i / 125; return {Eg, eff: 100 * shockleyQueisser(Eg, sq.Ts, sq.Tc, X).eff}; });
  const r = shockleyQueisser(sq.Eg, sq.Ts, sq.Tc, X), f = (6.957e8 / 1.496e11) ** 2 * X;
  // energy budget per unit area
  const flux = E => 2 * Math.PI / (h ** 3 * c0 ** 2) * (E * qe) ** 2 / Math.expm1(E / kT(sq.Ts)) * qe;
  let Nabs = 0, Pabs = 0; const n = 2000, Emax = sq.Eg + 40 * kT(sq.Ts), dE = (Emax - sq.Eg) / n;
  for (let i = 0; i < n; i++) { const E = sq.Eg + (i + 0.5) * dE; Nabs += f * flux(E) * dE; Pabs += f * flux(E) * E * qe * dE; }
  const Pin = r.Pin, below = Pin - Pabs, therm = Pabs - Nabs * sq.Eg * qe, Pout = r.eff * Pin, junction = Nabs * sq.Eg * qe - Pout;
  const best = curve.reduce((b, c) => c.eff > b.eff ? c : b, curve[0]);
  const budget = [{part: "not absorbed (hν < E_g)", v: below}, {part: "thermalisation (hν − E_g)", v: therm},
                  {part: "voltage + fill-factor loss (E_g − eV_m)", v: junction}, {part: "electrical output", v: Pout}].map(d => ({...d, pct: 100 * d.v / Pin}));
  return {curve, r, best, budget, X};
}
Plot.plot({
  width: Math.min(width, 1050), height: 280, x: {label: "band gap E_g (eV)"}, y: {label: "efficiency limit (%)", grid: true},
  marks: [Plot.line(sqOut.curve, {x: "Eg", y: "eff", stroke: "#2a78d6", strokeWidth: 2.5}),
          Plot.dot([{Eg: sq.Eg, eff: 100 * sqOut.r.eff}], {x: "Eg", y: "eff", r: 6, fill: "#e34948"}),
          Plot.text([{Eg: 1.12, eff: 3}, {Eg: 1.42, eff: 3}, {Eg: 0.66, eff: 3}, {Eg: 1.7, eff: 3}], {x: "Eg", y: "eff", text: d => ({1.12: "Si", 1.42: "GaAs", 0.66: "Ge", 1.7: "a-Si"})[d.Eg], fontSize: 10})]
})
```

```{ojs}
//| echo: false
Plot.plot({
  width: Math.min(width, 1050), height: 110, x: {label: "% of the incident power", domain: [0, 100]}, color: {legend: true, range: ["#c3c2b7", "#eb6834", "#eda100", "#1baf7a"]},
  marks: [Plot.barX(sqOut.budget, Plot.stackX({x: "pct", fill: "part", title: d => `${d.part}: ${d.pct.toFixed(1)} %`}))]
})
```

```{ojs}
//| echo: false
html`<div class="result-box">E<sub>g</sub> = ${sq.Eg} eV: J<sub>sc</sub> = ${(sqOut.r.Jsc / 10).toFixed(1)} mA/cm² · V<sub>oc</sub> = ${sqOut.r.Voc.toFixed(3)} V
(eV<sub>oc</sub>/E<sub>g</sub> = ${(sqOut.r.Voc / sq.Eg).toFixed(2)}) · FF = ${(100 * sqOut.r.FF).toFixed(1)} % · <b>η = ${(100 * sqOut.r.eff).toFixed(1)} %</b><br>
best band gap ${sqOut.best.Eg.toFixed(2)} eV → ${sqOut.best.eff.toFixed(1)} % · input ${sqOut.r.Pin.toFixed(0)} W/m² ·
losses: ${sqOut.budget.slice(0, 3).map(b => `${b.part.split(" (")[0]} ${b.pct.toFixed(1)} %`).join(", ")}</div>`
```

Things to try: for Si (1.12 eV) the limit is about 30 %; the record of 27 % (2025) is remarkably close. The two big losses are
complementary: below-gap photons dominate for wide gaps, thermalisation for narrow gaps. Concentrate the light (X = 1000): $J_{sc}$ grows
by 1000, $V_{oc}$ by $k_BT\ln1000=0.18$ V, and the limit rises to about 37 %; at the maximum concentration (the cell sees the Sun over the
full hemisphere) it reaches about 40 %. Heat the cell: $J_0$ grows, $V_{oc}$ falls.

## Recomputed: the classic numbers

In [ ]:
#| label: tbl-sq
#| tbl-cap: "Detailed-balance limit under 5778 K black-body sunlight (1 sun ≈ 1360 W/m²), compared with the records of slide 130. Real cells lose further voltage to non-radiative recombination (Ch. 3) and current to reflection and incomplete absorption."
rows = []
for name, Eg, rec in (("Ge", 0.66, None), ("Si", 1.11, 25.0), ("InP", 1.35, 22.1), ("GaAs", 1.42, 27.6), ("a-Si:H", 1.7, 10.1)):
    r = dl.shockley_queisser(Eg)
    rows.append((name, Eg, f"{r['Jsc']/10:.1f}", f"{r['Voc']:.3f}", f"{r['FF']:.3f}", f"{100*r['eff']:.1f}", "–" if rec is None else rec))
Egs = np.linspace(0.6, 2.5, 300); eff = np.array([dl.shockley_queisser(E)['eff'] for E in Egs])
rows.append(("optimum", round(Egs[eff.argmax()], 2), "", "", "", f"{100*eff.max():.1f}", ""))
pd.DataFrame(rows, columns=["material", "E_g (eV)", "J_sc (mA/cm²)", "V_oc (V)", "FF", "SQ limit (%)", "record, slide 130 (%)"])

For Si the black-body spectrum gives $J_{sc}\approx53$ mA/cm²; the real AM1.5 spectrum, depleted in the infrared by water and CO₂
absorption, gives 43.8 mA/cm², very close to the 42.7 mA/cm² of the PERL record cell: crystalline Si collects almost every photon. Its
voltage, 0.707 V instead of the radiative-limit 0.85–0.88 V, is where the remaining gap lies: Auger and surface recombination (Ch. 3)
replace the purely radiative $J_0$ of the ideal cell.

## Two junctions (slide 132)

In [ ]:
#| label: fig-tandem
#| fig-cap: "Detailed-balance efficiency of a four-terminal (independently connected) tandem: the top cell absorbs hν > E_g1, the bottom cell E_g2 < hν < E_g1. The optimum near (1.8, 0.93) eV gives about 42 %. The GaInP/GaAs pair of slide 132 (1.95/1.42 eV, ×) is well off the peak: its bottom gap is too wide and wastes the infrared, which is why the triple-junction cell adds a Ge (0.66 eV) cell underneath."
def cell_eff(Eg_lo, Eg_hi, Ts=5778.0, Tc=300.0):
    f = (6.957e8/1.496e11)**2
    E = np.linspace(Eg_lo, Eg_hi, 1500)
    Jsc = dl.qe*f*np.trapezoid(dl._photon_flux_bb(E, Ts), E)
    Ec = np.linspace(Eg_lo, Eg_lo + 40*dl.kT(Tc), 1500)
    J0 = dl.qe*np.trapezoid(dl._photon_flux_bb(Ec, Tc), Ec)
    V = np.linspace(0, dl.kT(Tc)*np.log(Jsc/J0 + 1), 800)
    return (V*(Jsc - J0*np.expm1(V/dl.kT(Tc)))).max()/(f*dl.sigma_SB*Ts**4)
g1 = np.linspace(1.3, 2.4, 34); g2 = np.linspace(0.6, 1.5, 28)
Z = np.array([[cell_eff(a, 6.0) + cell_eff(b, a) if a > b else np.nan for a in g1] for b in g2])
fig, ax = plt.subplots(figsize=(6.5, 4.4))
cs = ax.contourf(g1, g2, 100*Z, levels=np.arange(28, 47, 1), cmap="viridis"); fig.colorbar(cs, label="efficiency (%)")
i, j = np.unravel_index(np.nanargmax(Z), Z.shape); ax.plot(g1[j], g2[i], "w*", ms=12)
ax.plot(1.95, 1.42, "rx", ms=10, mew=2)
ax.set(xlabel="top-cell gap E_g1 (eV)", ylabel="bottom-cell gap E_g2 (eV)", title=f"max {100*np.nanmax(Z):.1f} % at ({g1[j]:.2f}, {g2[i]:.2f}) eV"); plt.show()

::: {.callout-tip collapse="true"}
## Going further
- **Infinite junctions.** With an infinite stack of cells, each converting a narrow slice of the spectrum, the limit rises to 68 % (one sun)
  and 86 % (full concentration). Extend the tandem code to $N$ cells with equal photon currents.
- **Photon recycling and external luminescence.** A real cell has non-radiative recombination: $V_{oc}=V_{oc}^{SQ}+\frac{k_BT}{e}\ln\eta_\text{ext}$,
  where $\eta_\text{ext}$ is the external luminescence efficiency. "A great solar cell must be a great LED" (Miller, Yablonovitch, 2012):
  connect this with the LED efficiency of Ch. 3. GaAs record cells have $\eta_\text{ext}\approx30$ %.
- **Hot-carrier cells and singlet fission** try to recover the thermalisation loss; **intermediate-band cells** and up-conversion try to use
  sub-gap photons. What do they need to beat 31 %?
- **Thermodynamic limit.** Even an ideal converter is limited by Carnot (1 − 300/5800 = 95 %) and, more strictly, by the Landsberg limit
  (93 %). Where does the remaining loss of the SQ cell come from in thermodynamic terms (entropy of the absorbed photons, étendue
  expansion from the solar disc to the hemisphere)?
:::

::: {.callout-note collapse="true"}
## Complements
- W. Shockley and H. J. Queisser, "Detailed balance limit of efficiency of p–n junction solar cells", *J. Appl. Phys.* 32, 510 (1961).
- O. D. Miller, E. Yablonovitch, S. R. Kurtz, "Strong internal and external luminescence as solar cells approach the Shockley–Queisser
  limit", *IEEE J. Photovolt.* 2, 303 (2012).
- S. Rühle, "Tabulated values of the Shockley–Queisser limit for single junction solar cells", *Sol. Energy* 130, 139 (2016) (AM1.5 values).
:::

## Try it yourself

1. Show that $J_0$ for Si at 300 K is of order $10^{-12}$ A/m² in the radiative limit, and compare with the $10^{-9}$–$10^{-8}$ A/m² of
   real Si cells. How many millivolts of $V_{oc}$ does that cost?
2. Why does concentrating the light raise the efficiency limit? Where does the extra voltage come from?
3. The limit depends on the temperature of the cell. By how much does the Si limit fall between 25 °C and 65 °C (a module on a roof in
   summer)?
4. Space solar cells see the AM0 spectrum (1361 W/m², close to the black body used here) while terrestrial cells see AM1.5 with absorption
   bands of water and CO₂. Why is the terrestrial limit *higher* (33.7 %) than the black-body value?